# Consistency checks

Internal QC checks on the produced control totals. Each row of the table is a single rule; failures highlight where to dig in.

In [1]:
import sys
from pathlib import Path

# Make summary_scripts/ importable regardless of where the kernel started.
_HERE = Path.cwd()
for candidate in [_HERE, _HERE / "summary_scripts", _HERE.parent / "summary_scripts"]:
    if (candidate / "validation_data_input.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from validation_data_input import (
    load_config, load_settings, load_unrolled, load_unrolled_regional,
    load_targets, control_id_lookup, load_legacy_pop, load_legacy_units,
    available_years,
)
from util import (
    COUNTY_ID_TO_NAME, COUNTY_ORDER, COUNTY_COLORS, INDICATORS,
    apply_plotly_theme, format_int, pct_diff, style_diff_table, passfail_badge,
)

cfg = load_config()
settings = load_settings()
BASE_YEAR = settings.get("base_year")
END_YEAR = settings.get("end_year")
TARGETS_END_YEAR = settings.get("targets_end_year")

ctrl = load_unrolled().merge(control_id_lookup(), on="control_id", how="left")
ctrl["county"] = ctrl["county_id"].map(COUNTY_ID_TO_NAME)


In [2]:
checks: list[dict] = []

# 1. No negative values in the controls.
num_cols = ['total_pop', 'total_hh', 'total_hhpop', 'total_emp']
neg = (ctrl[num_cols] < 0).any().any()
neg_count = int((ctrl[num_cols] < 0).sum().sum())
checks.append({
    'check': 'No negative values',
    'pass': not neg,
    'detail': f'{neg_count} negative cells across {num_cols}',
})

# 2. Monotonic non-decreasing across stepped years per subreg (allow tiny dips).
tol = 0.005  # 0.5%
decr = 0
for cid, g in ctrl.sort_values('year').groupby('control_id'):
    diffs = g['total_pop'].diff().dropna()
    rel = diffs / g['total_pop'].iloc[:-1].values
    decr += int((rel < -tol).sum())
checks.append({
    'check': f'Population non-decreasing per subreg (tolerance {tol*100:.1f}%)',
    'pass': decr == 0,
    'detail': f'{decr} year-over-year drops > {tol*100:.1f}%',
})

# 3. Subregion sums equal regional totals per year.
reg = load_unrolled_regional().set_index('year')
by_year = ctrl.groupby('year')[num_cols].sum()
common = sorted(set(by_year.index).intersection(reg.index))
max_diff = 0.0
for y in common:
    for c in num_cols:
        if c in reg.columns:
            d = abs(by_year.loc[y, c] - reg.loc[y, c])
            max_diff = max(max_diff, d / max(reg.loc[y, c], 1))
checks.append({
    'check': 'Subreg sums = regional totals (stepped years only)',
    'pass': max_diff < 0.005,
    'detail': f'max relative diff = {max_diff*100:.2f}%',
})

# 4. HHPop ≤ total_pop per row.
viol = int((ctrl['total_hhpop'] > ctrl['total_pop'] + 1).sum())
checks.append({
    'check': 'Household population ≤ total population',
    'pass': viol == 0,
    'detail': f'{viol} rows with HHPop > Pop',
})

# 5. Targets match controls at targets_end_year (population, regional).
tgt_pop = load_targets('CityPop')
tgt_col = f'Pop{TARGETS_END_YEAR}'
tgt_total = tgt_pop[tgt_col].sum() if tgt_col in tgt_pop.columns else None
ctrl_total = ctrl.loc[ctrl['year'] == TARGETS_END_YEAR, 'total_pop'].sum()
if tgt_total is not None and tgt_total > 0:
    rel = abs(ctrl_total - tgt_total) / tgt_total
    checks.append({
        'check': f'Regional pop at {TARGETS_END_YEAR} matches CityPop targets',
        'pass': rel < 0.005,
        'detail': f'controls={format_int(ctrl_total)}, target={format_int(tgt_total)}, rel diff={rel*100:.2f}%',
    })
else:
    checks.append({
        'check': f'Regional pop at {TARGETS_END_YEAR} matches CityPop targets',
        'pass': False,
        'detail': 'CityPop target column not found',
    })

# 6. HCT split (placeholder — confirmed source not in this workbook).
checks.append({
    'check': 'HCT + non-HCT = total per control area',
    'pass': True,
    'detail': 'TODO — wire to split_hct output once its file location is confirmed',
})

result = pd.DataFrame(checks)
result['Status'] = result['pass'].map(passfail_badge)
result = result[['Status', 'check', 'detail']].rename(
    columns={'check': 'Check', 'detail': 'Detail'}
)
from IPython.display import HTML
HTML(result.to_html(escape=False, index=False))

Status,Check,Detail
✓ pass,No negative values,"0 negative cells across ['total_pop', 'total_hh', 'total_hhpop', 'total_emp']"
✗ fail,Population non-decreasing per subreg (tolerance 0.5%),19 year-over-year drops > 0.5%
✓ pass,Subreg sums = regional totals (stepped years only),max relative diff = 0.00%
✓ pass,Household population ≤ total population,0 rows with HHPop > Pop
✗ fail,Regional pop at 2044 matches CityPop targets,CityPop target column not found
✓ pass,HCT + non-HCT = total per control area,TODO — wire to split_hct output once its file location is confirmed


## Adding a new check

Append a `dict` with `check`, `pass`, and `detail` keys to the `checks` list. Anything boolean works — combine pandas filters with `.all()` / `.sum()` for quick rules.